# Глава 5: Форматирование вывода и выступление от имени Клода

- [Урок](#lesson)
- [Упражнения](#exercises)
- [Пример игровой площадки](#example-playground)

## Настройка

Запустите следующую ячейку настройки, чтобы загрузить ключ API и установить вспомогательную функцию get_completion.

In [ ]:
# Import python's built-in regular expression library
import re
import boto3
import json

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Retrieve the MODEL_NAME variable from the IPython store
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages":[
              {"role": "user", "content": prompt},
              {"role": "assistant", "content": prefill}
            ],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')

---

## Урок

**Claude может форматировать вывод самыми разными способами**. Вам просто нужно об этом попросить!

Один из этих способов — использование тегов XML для отделения ответа от любого другого лишнего текста. Вы уже узнали, что можете использовать теги XML, чтобы сделать приглашение более понятным и понятным для Клода. Оказывается, вы также можете попросить Клода **использовать теги XML, чтобы сделать вывод более ясным и понятным** для людей.

### Примеры

Помните «проблему с преамбулой стихотворения», которую мы решили в главе 2, попросив Клода полностью пропустить преамбулу? Оказывается, мы можем добиться аналогичного результата, **сказав Клоду поместить стихотворение в теги XML**.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Почему мы хотим это сделать? Что ж, наличие вывода в **тегах XML позволяет конечному пользователю надежно получить стихотворение и только стихотворение, написав короткую программу для извлечения содержимого между тегами XML**.

Расширением этого метода является **помещение первого тега XML в очередь `assistant`. Когда вы помещаете текст в очередь «assistant», вы по сути сообщаете Клоду, что Клод уже что-то сказал и что с этого момента следует продолжать. Этот прием называется «говорить от имени Клода» или «предварительно заполнять ответ Клода».

Ниже мы сделали это с первым XML-тегом <haiku>. Обратите внимание, как Клод продолжает с того места, на котором мы остановились.

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Клод также преуспевает в использовании других стилей форматирования вывода, в частности JSON. Если вы хотите обеспечить вывод JSON (не детерминированно, но близко к этому), вы также можете предварительно заполнить ответ Клода открывающей скобкой `{`}.

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Ниже приведен пример **нескольких входных переменных в одной и той же спецификации приглашения И форматирования вывода, причем все они выполняются с использованием тегов XML**.

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

#### Бонусный урок

Если вы вызываете Claude через API, вы можете передать закрывающий XML-тег в параметр stop_sequences, чтобы Claude прекратил выборку после того, как он выдаст нужный тег. Это может сэкономить деньги и время до последнего токена, исключив заключительные замечания Клода после того, как он уже дал вам интересующий вас ответ.

Если вы хотите поэкспериментировать с подсказками к уроку, не меняя никакого содержания выше, прокрутите блокнот урока до конца, чтобы перейти к [**Пример игровой площадки**](#example-playground).

---

## Упражнения
- [Упражнение 5.1 – КОЗА Стефа Карри](#exercise-51---steph-curry-goat)
- [Упражнение 5.2 – Два хайку](#exercise-52---two-haikus)
- [Упражнение 5.3 – Два хайку, два животных](#exercise-53---two-haikus-two-animals)

### Упражнение 5.1 — Стеф Карри КОЗА
Вынужденный сделать выбор, Клод называет Майкла Джордана лучшим баскетболистом всех времен. Можем ли мы заставить Клода выбрать кого-нибудь другого?

Измените переменную `PREFILL`, чтобы **заставить Клода привести подробный аргумент о том, что лучшим баскетболистом всех времён является Стивен Карри**. Постарайтесь не менять ничего, кроме «PREFILL», поскольку это основная задача данного упражнения.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Prefill for Claude's response
PREFILL = ""

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
print(hints.exercise_5_1_hint)

### Упражнение 5.2 — Два хайку
Измените приведенный ниже код `PROMPT`, используя теги XML, чтобы Клод написал два хайку о животном вместо одного. Должно быть ясно, где заканчивается одно стихотворение и начинается другое.

In [ ]:
# Variable content
ANIMAL = "cats"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
print(hints.exercise_5_2_hint)

### Упражнение 5.3 — Два хайку, два животных
Измените приведенный ниже код «PROMPT», чтобы **Клод создавал два хайку о двух разных животных**. Используйте {ANIMAL1} в качестве замены для первой замены и {ANIMAL2} в качестве замены для второй замены.

In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
print(hints.exercise_5_3_hint)

### Поздравляю!

Если вы выполнили все упражнения до этого момента, вы готовы перейти к следующей главе. Приятного подсказки!

---

## Пример игровой площадки

Это область, где вы можете свободно экспериментировать с примерами подсказок, представленными в этом уроке, и настраивать подсказки, чтобы увидеть, как они могут повлиять на ответы Клода.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))